# DEPICT prediction notebook: validation and test predicted DGE

This notebook is an interactive, provenance-preserving notebook version of `prediction.py`. For each selected fold, it total-normalizes the source AnnData, loads that fold's trained checkpoint, and writes **predicted DGE** (`Pred_delta = predicted expression − paired baseline`) for both validation and test perturbations.

The outputs are stored as HDF5, rather than CSV with stringified arrays, to preserve numeric matrices and avoid extremely large, slow CSV files. Each output also includes `adata_row`, `obs_name`, `pert_iname`, `cell_id`, dose/time, baseline, observed DGE, and predicted DGE.

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import h5py
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import pearsonr
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# ---------------------------------------------------------------------
# Project defaults
# ---------------------------------------------------------------------
PROJECT_ROOT = Path("~/DEPICT")
MAIN_WORK_DIR = PROJECT_ROOT / "Code/downstream_analysis_code/TransferabilityUnseenCell"
TRAIN_CODE_DIR = PROJECT_ROOT / "Code/train_code"

DEFAULT_ADATA = PROJECT_ROOT / "Data/FinalData/adataAfterClean.h5ad"
DEFAULT_LLM = PROJECT_ROOT / "Data/FinalData/gptEmbed_Jul9_final.csv"
DEFAULT_MFP = PROJECT_ROOT / "Data/FinalData/compounds_512MFP_wholeDat_fixed.csv"
DEFAULT_DRUG_TARGETS = PROJECT_ROOT / "Data/FinalData/compounds_target_multihot_full.csv"
DEFAULT_MODEL_DIR = PROJECT_ROOT / "Model"
DEFAULT_TRAINING_CODE = (
    TRAIN_CODE_DIR /
    "Transformer_d32h8l4_dp1_MSECor_lr1CosSche_3XAttn_sepGeneEncPred_"
    "newAtten_FiLM_Enc3DimReducLa128_CellAware_frontEndMLPSimp_"
    "DoseTimeAsScalarXattns_LLMMFPonly_rest150Epoch.py"
)

RANDOM_SPLITS = [f"random_split{i}" for i in range(1, 6)]
CELL_SPLITS = [f"cell_split{i}" for i in range(1, 6)]
DRUG_SPLITS = [f"drug_split{i}" for i in range(1, 6)]
ALL_SPLITS = RANDOM_SPLITS + CELL_SPLITS + DRUG_SPLITS

CHECKPOINT_TEMPLATE = (
    "transformer_d32h8l4_{split_type}_dp1_MSECor_lr1_CosSche_"
    "3XAttn_sepGene2EncPred_newAttn_FiLM_Enc3DimReducLa128_"
    "CellAware_frontEndMLPsimp_whole_DoseTimeAsScalarXattns_"
    "LLMMFP_first50epoch.pth"
)

N_GENES = 978
LLM_DIM = 512
MFP_DIM = 512


def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)


def hash_sequence(values: Iterable[object]) -> str:
    digest = hashlib.sha256()
    for value in values:
        encoded = str(value).encode("utf-8")
        digest.update(len(encoded).to_bytes(8, byteorder="big"))
        digest.update(encoded)
    return digest.hexdigest()


def resolve_device(device_text: str) -> torch.device:
    if device_text == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device(device_text)
    if device.type == "cuda":
        require(torch.cuda.is_available(), "CUDA was requested but is unavailable.")
    return device


def to_dense_1d(row) -> np.ndarray:
    return row.toarray().ravel() if hasattr(row, "toarray") else np.asarray(row).ravel()


def validate_feature_table(df: pd.DataFrame, label: str, expected_dim: int) -> None:
    require(df.index.is_unique, f"{label} has duplicate drug identifiers.")
    require(df.columns.is_unique, f"{label} has duplicate columns.")
    require(df.shape[1] == expected_dim, f"{label} has {df.shape[1]} columns; expected {expected_dim}.")


In [2]:
# ---------------------------------------------------------------------
# Exact model-class extraction from the supplied original training source.
# This avoids importing the source file and accidentally executing training.
# ---------------------------------------------------------------------
def load_model_class_from_training_source(training_code_path: Path) -> type[nn.Module]:
    require(training_code_path.exists(), f"Training source is missing: {training_code_path}")

    source_text = training_code_path.read_text(encoding="utf-8")
    tree = ast.parse(source_text, filename=str(training_code_path))

    needed = {"Enc3Layer", "GenePerturbationTransformer"}
    nodes = [
        node for node in tree.body
        if isinstance(node, ast.ClassDef) and node.name in needed
    ]
    found = {node.name for node in nodes}
    require(found == needed, f"Could not extract model classes: missing {sorted(needed - found)}.")

    module = ast.Module(body=nodes, type_ignores=[])
    ast.fix_missing_locations(module)

    namespace = {"torch": torch, "nn": nn, "F": F}
    exec(compile(module, filename=str(training_code_path), mode="exec"), namespace)
    return namespace["GenePerturbationTransformer"]


def load_fold_model(
    model_class: type[nn.Module],
    checkpoint_path: Path,
    device: torch.device,
) -> tuple[nn.Module, dict]:
    require(checkpoint_path.exists(), f"Checkpoint missing: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    required_keys = {"model_state", "optim_state", "sched_state", "epoch", "best_val_mse"}
    require(
        required_keys.issubset(checkpoint.keys()),
        f"Checkpoint missing keys: {sorted(required_keys - set(checkpoint.keys()))}",
    )

    # Explicit training-time instantiation arguments, not the class defaults.
    model = model_class(
        d_model=32,
        num_heads=8,
        num_encoder_layers=4,
        dropout=0.1,
        max_len=978,
        d_hidden=64,
        n_genes_target=1225,
        fp_bits=512,
        llm_dim=512,
        ae_latent=128,
    ).to(device)

    model.load_state_dict(checkpoint["model_state"], strict=True)
    model.eval()
    return model, checkpoint


In [3]:
# ---------------------------------------------------------------------
# Dataset: same prediction.py logic, with row metadata retained explicitly.
# ---------------------------------------------------------------------
class DEPICTPredictionDataset(Dataset):
    def __init__(
        self,
        adata,
        gpt_embed_df: pd.DataFrame,
        mfp_df: pd.DataFrame,
        drug_targets: pd.DataFrame,
        split_type: str,
        subset: str,
    ):
        require(subset in {"valid", "test"}, "subset must be 'valid' or 'test'.")
        require(split_type in adata.obs.columns, f"AnnData does not contain split column {split_type}.")
        require("paired_control_index" in adata.obs.columns, "Missing paired_control_index.")
        require(adata.obs_names.is_unique, "adata.obs_names must be unique.")

        self.adata = adata
        self.gpt_embed_df = gpt_embed_df
        self.mfp_df = mfp_df
        self.drug_targets = drug_targets
        self.split_type = split_type
        self.subset = subset
        self.indices = np.where(adata.obs[split_type].astype(str).to_numpy() == subset)[0]
        require(len(self.indices) > 0, f"No {subset} rows for {split_type}.")

        # Same source logic: per-cell control mean/variance after total normalization.
        control_mask = adata.obs["control"].to_numpy() == 1
        control_x = adata.X[control_mask]
        control_x = control_x.toarray() if hasattr(control_x, "toarray") else np.asarray(control_x)
        control_x = np.asarray(control_x, dtype=np.float32)
        control_cells = adata.obs.loc[control_mask, "cell_id"].astype(str).to_numpy()

        self.cell_stats = {}
        for cell_id in np.unique(control_cells):
            x_cell = control_x[control_cells == cell_id]
            mu = x_cell.mean(axis=0, dtype=np.float32)
            var = x_cell.var(axis=0, dtype=np.float32)
            self.cell_stats[cell_id] = np.concatenate([mu, var]).astype(np.float32, copy=False)

        subset_drugs = set(adata.obs.iloc[self.indices]["pert_iname"].astype(str))
        require(subset_drugs.issubset(set(gpt_embed_df.index.astype(str))), f"{split_type}/{subset}: missing LLM features.")
        require(subset_drugs.issubset(set(mfp_df.index.astype(str))), f"{split_type}/{subset}: missing MFP features.")
        require(subset_drugs.issubset(set(drug_targets.index.astype(str))), f"{split_type}/{subset}: missing target features.")

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int):
        adata_row = int(self.indices[index])
        obs = self.adata.obs.iloc[adata_row]

        paired_control_id = str(obs["paired_control_index"])
        require(
            paired_control_id in self.adata.obs_names,
            f"Missing paired control {paired_control_id} for row {self.adata.obs_names[adata_row]}.",
        )
        paired_row = int(self.adata.obs_names.get_loc(paired_control_id))

        baseline = to_dense_1d(self.adata.X[paired_row]).astype(np.float32, copy=False)
        observed = to_dense_1d(self.adata.X[adata_row]).astype(np.float32, copy=False)

        control_cell = str(self.adata.obs.iloc[paired_row]["cell_id"])
        drug_name = str(obs["pert_iname"])

        dose = float(obs["dose"])
        pert_time = float(obs["pert_time"])
        require(dose >= 0 and pert_time >= 0, f"Negative dose/time at {self.adata.obs_names[adata_row]}.")

        return {
            "adata_row": np.int64(adata_row),
            "x_base": torch.from_numpy(baseline),
            "cell_stats": torch.from_numpy(self.cell_stats[control_cell]),
            "x_llm": torch.from_numpy(self.gpt_embed_df.loc[drug_name].to_numpy(dtype=np.float32, copy=True)),
            "x_mfp": torch.from_numpy(self.mfp_df.loc[drug_name].to_numpy(dtype=np.float32, copy=True)),
            "x_tgt": torch.from_numpy(self.drug_targets.loc[drug_name].to_numpy(dtype=np.float32, copy=True)),
            "y_true": torch.from_numpy(observed),
            "dose_feat": torch.tensor(math.log10(dose + 1.0), dtype=torch.float32),
            "time_feat": torch.tensor(math.log10(pert_time + 1.0), dtype=torch.float32),
        }


In [ ]:
# ---------------------------------------------------------------------
# Interactive configuration — edit before running any prediction cell.
# ---------------------------------------------------------------------
BASE_CONFIG = {
    "adata_path": DEFAULT_ADATA,
    "llm_path": DEFAULT_LLM,
    "mfp_path": DEFAULT_MFP,
    "drug_target_path": DEFAULT_DRUG_TARGETS,
    "model_dir": DEFAULT_MODEL_DIR,
    "training_code_path": DEFAULT_TRAINING_CODE,
    "output_dir": MAIN_WORK_DIR / "predicted_dge",

    "batch_size": 128,
    "num_workers": 0,
    "device": "auto",       # "auto", "cpu", "cuda", or "cuda:0"
    "compression": "gzip",  # "gzip", "lzf", or "none"
    "overwrite": False,
}

# Fold groups. Each of the six run cells below uses exactly one of these groups.
RANDOM_SPLIT_CONFIG = {**BASE_CONFIG, "split_types": RANDOM_SPLITS}
CELL_SPLIT_CONFIG = {**BASE_CONFIG, "split_types": CELL_SPLITS}
DRUG_SPLIT_CONFIG = {**BASE_CONFIG, "split_types": DRUG_SPLITS}

# Optional smoke-test examples:
# RANDOM_SPLIT_CONFIG = {**BASE_CONFIG, "split_types": ["random_split1"]}
# CELL_SPLIT_CONFIG   = {**BASE_CONFIG, "split_types": ["cell_split1"]}
# DRUG_SPLIT_CONFIG   = {**BASE_CONFIG, "split_types": ["drug_split1"]}


In [5]:
# ---------------------------------------------------------------------
# Prediction, HDF5 writing, and statistics exactly aligned with prediction.py.
# ---------------------------------------------------------------------
def correlation_loss(output: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    vx = output - output.mean(dim=1, keepdim=True)
    vy = target - target.mean(dim=1, keepdim=True)
    corr = (vx * vy).sum(dim=1) / (
        torch.norm(vx, dim=1) * torch.norm(vy, dim=1) + 1e-8
    )
    return 1 - corr.mean()


def mse_plus_correlation(output: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    # This is the exact "MSE+Correlation" loss used by prediction.py.
    return nn.MSELoss()(output, target) + 0.3 * (correlation_loss(output, target) - 1)


def create_prediction_h5_datasets(h5: h5py.File, n_rows: int, batch_size: int, compression):
    text_dtype = h5py.string_dtype(encoding="utf-8")
    chunk_rows = min(max(1, batch_size), n_rows)
    gene_chunks = (chunk_rows, N_GENES)

    for key, dtype in [
        ("adata_row", np.int64),
        ("obs_name", text_dtype),
        ("pert_iname", text_dtype),
        ("cell_id", text_dtype),
        ("dose", np.float32),
        ("pert_time", np.float32),
    ]:
        h5.create_dataset(key, shape=(n_rows,), dtype=dtype)

    for key in [
        "baseline",
        "observed_expression",
        "predicted_expression",
        "observed_dge",
        "predicted_dge",
    ]:
        h5.create_dataset(
            key,
            shape=(n_rows, N_GENES),
            dtype=np.float32,
            chunks=gene_chunks,
            compression=compression,
            shuffle=True,
        )


def summarize_statistics_like_prediction_py(
    all_true: list[torch.Tensor],
    all_pred: list[torch.Tensor],
    all_true_delta: list[torch.Tensor],
    all_pred_delta: list[torch.Tensor],
    running_loss: float,
    n_rows: int,
) -> dict:
    """
    Reproduces prediction.py eval_model output:
      epoch_loss, avg_r2, mse_loss, avg_pcc, avg_delta_r2, avg_delta_pcc.
    """
    all_y_true = torch.cat(all_true, dim=0)
    all_y_pred = torch.cat(all_pred, dim=0)
    all_delta_true = torch.cat(all_true_delta, dim=0)
    all_delta_pred = torch.cat(all_pred_delta, dim=0)

    r2_list, pcc_list = [], []
    r2_delta_list, pcc_delta_list = [], []

    for i in range(all_y_true.shape[0]):
        true_sample = all_y_true[i].numpy()
        pred_sample = all_y_pred[i].numpy()
        true_delta = all_delta_true[i].numpy()
        pred_delta = all_delta_pred[i].numpy()

        r2_list.append(r2_score(true_sample, pred_sample))
        r2_delta_list.append(r2_score(true_delta, pred_delta))

        if np.std(true_sample) > 1e-6 and np.std(pred_sample) > 1e-6:
            pcc_list.append(float(pearsonr(true_sample, pred_sample)[0]))
        else:
            pcc_list.append(0.0)

        if np.std(true_delta) > 1e-6 and np.std(pred_delta) > 1e-6:
            pcc_delta_list.append(float(pearsonr(true_delta, pred_delta)[0]))
        else:
            pcc_delta_list.append(0.0)

    return {
        "loss": float(running_loss / n_rows),
        "mse": float(torch.mean((all_y_true - all_y_pred) ** 2).item()),
        "r2": float(np.mean(r2_list)),
        "pcc": float(np.mean(pcc_list)),
        "delta_r2": float(np.mean(r2_delta_list)),
        "delta_pcc": float(np.mean(pcc_delta_list)),
    }


@torch.no_grad()
def write_fold_subset_predictions_and_statistics(
    *,
    model: nn.Module,
    checkpoint: dict,
    dataset: DEPICTPredictionDataset,
    adata,
    out_path: Path,
    split_type: str,
    subset: str,
    device: torch.device,
    batch_size: int,
    num_workers: int,
    compression,
    overwrite: bool,
):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        if not overwrite:
            print(f"Output exists and overwrite=False; statistics will still be recomputed: {out_path}")
            write_output = False
            h5 = None
        else:
            out_path.unlink()
            write_output = True
            h5 = h5py.File(out_path, "w")
    else:
        write_output = True
        h5 = h5py.File(out_path, "w")

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )

    if write_output:
        h5.attrs["split_type"] = split_type
        h5.attrs["subset"] = subset
        h5.attrs["checkpoint_epoch"] = int(checkpoint["epoch"])
        h5.attrs["checkpoint_best_val_mse"] = float(checkpoint["best_val_mse"])
        h5.attrs["model_mode"] = "eval"
        h5.attrs["adata_var_names_hash_sha256"] = hash_sequence(adata.var_names.astype(str))
        h5.attrs["n_genes"] = N_GENES
        h5.attrs["normalization"] = "scanpy.pp.normalize_total(adata), default target_sum=None"
        h5.attrs["prediction_py_statistics_loss"] = "MSE + 0.3 * (correlation_loss - 1)"
        h5.attrs["predicted_dge_definition"] = "predicted_expression - paired_baseline"
        h5.attrs["observed_dge_definition"] = "observed_expression - paired_baseline"
        create_prediction_h5_datasets(
            h5=h5,
            n_rows=len(dataset),
            batch_size=batch_size,
            compression=compression,
        )

    running_loss = 0.0
    all_true, all_pred = [], []
    all_true_delta, all_pred_delta = [], []
    cursor = 0

    for batch_no, batch in enumerate(loader, start=1):
        x_base = batch["x_base"].to(device, non_blocking=True)
        cell_stats = batch["cell_stats"].to(device, non_blocking=True)
        x_llm = batch["x_llm"].to(device, non_blocking=True)
        x_mfp = batch["x_mfp"].to(device, non_blocking=True)
        x_tgt = batch["x_tgt"].to(device, non_blocking=True)
        y_true = batch["y_true"].to(device, non_blocking=True)
        dose_feat = batch["dose_feat"].to(device, non_blocking=True)
        time_feat = batch["time_feat"].to(device, non_blocking=True)

        y_pred = model(x_base, cell_stats, x_llm, x_mfp, x_tgt, dose_feat, time_feat)
        predicted_dge = y_pred - x_base
        observed_dge = y_true - x_base

        # Exactly like prediction.py eval_model: criterion is on delta.
        loss = mse_plus_correlation(predicted_dge, observed_dge)
        running_loss += float(loss.item()) * x_base.size(0)

        all_true.append(y_true.cpu())
        all_pred.append(y_pred.cpu())
        all_true_delta.append(observed_dge.cpu())
        all_pred_delta.append(predicted_dge.cpu())

        if write_output:
            n_batch = x_base.shape[0]
            end = cursor + n_batch
            rows = batch["adata_row"].cpu().numpy().astype(np.int64, copy=False)

            h5["adata_row"][cursor:end] = rows
            h5["obs_name"][cursor:end] = np.asarray(adata.obs_names[rows].astype(str), dtype=object)
            h5["pert_iname"][cursor:end] = np.asarray(
                adata.obs.iloc[rows]["pert_iname"].astype(str),
                dtype=object,
            )
            h5["cell_id"][cursor:end] = np.asarray(
                adata.obs.iloc[rows]["cell_id"].astype(str),
                dtype=object,
            )
            h5["dose"][cursor:end] = adata.obs.iloc[rows]["dose"].to_numpy(dtype=np.float32)
            h5["pert_time"][cursor:end] = adata.obs.iloc[rows]["pert_time"].to_numpy(dtype=np.float32)
            h5["baseline"][cursor:end] = x_base.cpu().numpy().astype(np.float32, copy=False)
            h5["observed_expression"][cursor:end] = y_true.cpu().numpy().astype(np.float32, copy=False)
            h5["predicted_expression"][cursor:end] = y_pred.cpu().numpy().astype(np.float32, copy=False)
            h5["observed_dge"][cursor:end] = observed_dge.cpu().numpy().astype(np.float32, copy=False)
            h5["predicted_dge"][cursor:end] = predicted_dge.cpu().numpy().astype(np.float32, copy=False)
            cursor = end

        if batch_no % 100 == 0 or batch_no == len(loader):
            print(f"  {split_type}/{subset}: processed {min(batch_no * batch_size, len(dataset)):,}/{len(dataset):,} rows")

    stats = summarize_statistics_like_prediction_py(
        all_true=all_true,
        all_pred=all_pred,
        all_true_delta=all_true_delta,
        all_pred_delta=all_pred_delta,
        running_loss=running_loss,
        n_rows=len(dataset),
    )

    if write_output:
        h5.attrs["statistics_loss"] = stats["loss"]
        h5.attrs["statistics_mse"] = stats["mse"]
        h5.attrs["statistics_r2"] = stats["r2"]
        h5.attrs["statistics_pcc"] = stats["pcc"]
        h5.attrs["statistics_delta_r2"] = stats["delta_r2"]
        h5.attrs["statistics_delta_pcc"] = stats["delta_pcc"]
        h5.close()

    return {
        "split_type": split_type,
        "subset": subset,
        "n_rows": int(len(dataset)),
        "checkpoint_epoch": int(checkpoint["epoch"]),
        "checkpoint_best_val_mse": float(checkpoint["best_val_mse"]),
        "output_status": "WRITTEN" if write_output else "EXISTING_NOT_OVERWRITTEN",
        "h5": str(out_path),
        **stats,
    }


In [6]:
# ---------------------------------------------------------------------
# Reusable runner. The TEST and VALIDATION cells below call this separately.
# ---------------------------------------------------------------------
def run_one_subset_prediction_extraction(config: dict) -> pd.DataFrame:
    require(config["subset"] in {"valid", "test"}, "CONFIG['subset'] must be 'valid' or 'test'.")
    require(config["batch_size"] > 0, "batch_size must be positive.")
    require(config["num_workers"] >= 0, "num_workers must be nonnegative.")
    require(set(config["split_types"]).issubset(set(ALL_SPLITS)), "Unknown split type in CONFIG.")

    device = resolve_device(config["device"])
    compression = None if config["compression"] == "none" else config["compression"]

    for path, label in [
        (config["adata_path"], "AnnData"),
        (config["llm_path"], "LLM feature table"),
        (config["mfp_path"], "MFP feature table"),
        (config["drug_target_path"], "drug-target table"),
        (config["model_dir"], "model directory"),
        (config["training_code_path"], "training source"),
    ]:
        require(Path(path).exists(), f"{label} not found: {path}")

    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Using device: {device}")
    print("Loading source inputs and applying training-time total normalization...")
    adata = sc.read(config["adata_path"])
    require(adata.n_vars == N_GENES, f"AnnData has {adata.n_vars} genes; expected {N_GENES}.")
    sc.pp.normalize_total(adata)

    llm_df = pd.read_csv(config["llm_path"], index_col=0)
    mfp_df = pd.read_csv(config["mfp_path"], index_col=0)
    drug_targets = pd.read_csv(config["drug_target_path"], index_col=0)

    validate_feature_table(llm_df, "LLM table", LLM_DIM)
    validate_feature_table(mfp_df, "MFP table", MFP_DIM)
    require(drug_targets.index.is_unique, "drug_targets index has duplicate rows.")

    ModelClass = load_model_class_from_training_source(Path(config["training_code_path"]))
    subset = config["subset"]
    results = []

    for split_type in config["split_types"]:
        print(f"\n{'=' * 92}\nProcessing {split_type} / {subset}\n{'=' * 92}")
        checkpoint_path = Path(config["model_dir"]) / CHECKPOINT_TEMPLATE.format(split_type=split_type)
        model, checkpoint = load_fold_model(ModelClass, checkpoint_path, device)

        dataset = DEPICTPredictionDataset(
            adata=adata,
            gpt_embed_df=llm_df,
            mfp_df=mfp_df,
            drug_targets=drug_targets,
            split_type=split_type,
            subset=subset,
        )

        out_path = output_dir / split_type / f"predicted_dge_{subset}.h5"
        result = write_fold_subset_predictions_and_statistics(
            model=model,
            checkpoint=checkpoint,
            dataset=dataset,
            adata=adata,
            out_path=out_path,
            split_type=split_type,
            subset=subset,
            device=device,
            batch_size=config["batch_size"],
            num_workers=config["num_workers"],
            compression=compression,
            overwrite=config["overwrite"],
        )
        results.append(result)

        # Exact presentation format used by prediction.py.
        print(
            f"{split_type} / {subset} | "
            f"Test Loss: {result['loss']:.4f}, "
            f"Test MSE: {result['mse']:.4f}, "
            f"Test R2: {result['r2']:.4f}, "
            f"Test PCC: {result['pcc']:.4f}, "
            f"Test delta R2: {result['delta_r2']:.4f}, "
            f"Test delta PCC: {result['delta_pcc']:.4f}"
        )

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    stats_df = pd.DataFrame(results)
    stats_dir = output_dir / "statistics"
    stats_dir.mkdir(parents=True, exist_ok=True)

    csv_path = stats_dir / f"prediction_py_style_statistics_{subset}.csv"
    json_path = stats_dir / f"prediction_py_style_statistics_{subset}.json"
    stats_df.to_csv(csv_path, index=False)
    json_path.write_text(stats_df.to_json(orient="records", indent=2))

    print(f"\nPASS: {subset} prediction extraction completed.")
    print(f"Statistics CSV:  {csv_path}")
    print(f"Statistics JSON: {json_path}")
    return stats_df


## Test random-split predictions

Runs only the five `random-split_split1`–`random-split_split5` folds for the `test` subset, writes their predicted DGE files, and prints the `prediction.py`-style performance statistics.

In [7]:
# TEST RANDOM-SPLIT PREDICTIONS
test_random_config = {
    **RANDOM_SPLIT_CONFIG,
    "subset": "test",
}

test_random_statistics = run_one_subset_prediction_extraction(test_random_config)
test_random_statistics

Using device: cuda
Loading source inputs and applying training-time total normalization...

Processing random_split1 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  random_split1/test: processed 12,800/83,665 rows
  random_split1/test: processed 25,600/83,665 rows
  random_split1/test: processed 38,400/83,665 rows
  random_split1/test: processed 51,200/83,665 rows
  random_split1/test: processed 64,000/83,665 rows
  random_split1/test: processed 76,800/83,665 rows
  random_split1/test: processed 83,665/83,665 rows
random_split1 / test | Test Loss: 0.5852, Test MSE: 0.7773, Test R2: 0.8737, Test PCC: 0.9338, Test delta R2: 0.4072, Test delta PCC: 0.6403

Processing random_split2 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  random_split2/test: processed 12,800/83,665 rows
  random_split2/test: processed 25,600/83,665 rows
  random_split2/test: processed 38,400/83,665 rows
  random_split2/test: processed 51,200/83,665 rows
  random_split2/test: processed 64,000/83,665 rows
  random_split2/test: processed 76,800/83,665 rows
  random_split2/test: processed 83,665/83,665 rows
random_split2 / test | Test Loss: 0.5833, Test MSE: 0.7755, Test R2: 0.8740, Test PCC: 0.9340, Test delta R2: 0.4072, Test delta PCC: 0.6407

Processing random_split3 / test


...

/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  random_split4/test: processed 12,800/83,665 rows
  random_split4/test: processed 25,600/83,665 rows
  random_split4/test: processed 38,400/83,665 rows
  random_split4/test: processed 51,200/83,665 rows
  random_split4/test: processed 64,000/83,665 rows
  random_split4/test: processed 76,800/83,665 rows
  random_split4/test: processed 83,665/83,665 rows
random_split4 / test | Test Loss: 0.5923, Test MSE: 0.7841, Test R2: 0.8727, Test PCC: 0.9333, Test delta R2: 0.4049, Test delta PCC: 0.6393

Processing random_split5 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  random_split5/test: processed 12,800/83,665 rows
  random_split5/test: processed 25,600/83,665 rows
  random_split5/test: processed 38,400/83,665 rows
  random_split5/test: processed 51,200/83,665 rows
  random_split5/test: processed 64,000/83,665 rows
  random_split5/test: processed 76,800/83,665 rows
  random_split5/test: processed 83,665/83,665 rows
random_split5 / test | Test Loss: 0.5897, Test MSE: 0.7817, Test R2: 0.8729, Test PCC: 0.9334, Test delta R2: 0.4058, Test delta PCC: 0.6400

PASS: test prediction extraction completed.
Statistics CSV:  /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_test.csv
Statistics JSON: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_test.json


,split_type,subset,n_rows,checkpoint_epoch,checkpoint_best_val_mse,output_status,h5,loss,mse,r2,pcc,delta_r2,delta_pcc
0,random_split1,test,83665,191,0.775185,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.585197,0.777289,0.873717,0.933813,0.407196,0.640308
1,random_split2,test,83665,192,0.780796,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.583304,0.775506,0.874023,0.933989,0.407223,0.640672
2,random_split3,test,83665,191,0.773551,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.582928,0.775459,0.874239,0.934113,0.408975,0.641769
3,random_split4,test,83665,200,0.775581,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.592305,0.784082,0.872712,0.933292,0.404930,0.639256
4,random_split5,test,83665,192,0.783841,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.589728,0.781733,0.872947,0.933417,0.405779,0.640019


## Test cell-split predictions

Runs only the five `cell-split_split1`–`cell-split_split5` folds for the `test` subset, writes their predicted DGE files, and prints the `prediction.py`-style performance statistics.

In [8]:
# TEST CELL-SPLIT PREDICTIONS
test_cell_config = {
    **CELL_SPLIT_CONFIG,
    "subset": "test",
}

test_cell_statistics = run_one_subset_prediction_extraction(test_cell_config)
test_cell_statistics

Using device: cuda
Loading source inputs and applying training-time total normalization...

Processing cell_split1 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  cell_split1/test: processed 12,510/12,510 rows
cell_split1 / test | Test Loss: 0.9514, Test MSE: 1.1199, Test R2: 0.8351, Test PCC: 0.9128, Test delta R2: 0.2532, Test delta PCC: 0.5618

Processing cell_split2 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  cell_split2/test: processed 12,800/25,742 rows
  cell_split2/test: processed 25,600/25,742 rows
  cell_split2/test: processed 25,742/25,742 rows
cell_split2 / test | Test Loss: 0.8304, Test MSE: 1.0033, Test R2: 0.8406, Test PCC: 0.9160, Test delta R2: 0.3138, Test delta PCC: 0.5762

Processing cell_split3 / test


...

/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  cell_split4/test: processed 12,800/137,725 rows
  cell_split4/test: processed 25,600/137,725 rows
  cell_split4/test: processed 38,400/137,725 rows
  cell_split4/test: processed 51,200/137,725 rows
  cell_split4/test: processed 64,000/137,725 rows
  cell_split4/test: processed 76,800/137,725 rows
  cell_split4/test: processed 89,600/137,725 rows
  cell_split4/test: processed 102,400/137,725 rows
  cell_split4/test: processed 115,200/137,725 rows
  cell_split4/test: processed 128,000/137,725 rows
  cell_split4/test: processed 137,725/137,725 rows
cell_split4 / test | Test Loss: 0.7066, Test MSE: 0.8770, Test R2: 0.8529, Test PCC: 0.9232, Test delta R2: 0.2986, Test delta PCC: 0.5679

Processing cell_split5 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  cell_split5/test: processed 12,800/86,370 rows
  cell_split5/test: processed 25,600/86,370 rows
  cell_split5/test: processed 38,400/86,370 rows
  cell_split5/test: processed 51,200/86,370 rows
  cell_split5/test: processed 64,000/86,370 rows
  cell_split5/test: processed 76,800/86,370 rows
  cell_split5/test: processed 86,370/86,370 rows
cell_split5 / test | Test Loss: 0.8437, Test MSE: 1.0134, Test R2: 0.8409, Test PCC: 0.9166, Test delta R2: 0.2633, Test delta PCC: 0.5659

PASS: test prediction extraction completed.
Statistics CSV:  /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_test.csv
Statistics JSON: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_test.json


,split_type,subset,n_rows,checkpoint_epoch,checkpoint_best_val_mse,output_status,h5,loss,mse,r2,pcc,delta_r2,delta_pcc
0,cell_split1,test,12510,96,0.920025,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.951380,1.119907,0.835111,0.912778,0.253187,0.561756
1,cell_split2,test,25742,88,0.966656,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.830409,1.003270,0.840639,0.916012,0.313767,0.576205
2,cell_split3,test,156258,45,0.914361,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.776616,0.945232,0.851440,0.922036,0.285594,0.562051
3,cell_split4,test,137725,36,0.999096,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.706618,0.876995,0.852927,0.923156,0.298607,0.567923
4,cell_split5,test,86370,85,0.893537,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.843656,1.013436,0.840924,0.916643,0.263270,0.565933


## Test drug-split predictions

Runs only the five `drug-split_split1`–`drug-split_split5` folds for the `test` subset, writes their predicted DGE files, and prints the `prediction.py`-style performance statistics.

In [9]:
# TEST DRUG-SPLIT PREDICTIONS
test_drug_config = {
    **DRUG_SPLIT_CONFIG,
    "subset": "test",
}

test_drug_statistics = run_one_subset_prediction_extraction(test_drug_config)
test_drug_statistics

Using device: cuda
Loading source inputs and applying training-time total normalization...

Processing drug_split1 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  drug_split1/test: processed 12,800/85,539 rows
  drug_split1/test: processed 25,600/85,539 rows
  drug_split1/test: processed 38,400/85,539 rows
  drug_split1/test: processed 51,200/85,539 rows
  drug_split1/test: processed 64,000/85,539 rows
  drug_split1/test: processed 76,800/85,539 rows
  drug_split1/test: processed 85,539/85,539 rows
drug_split1 / test | Test Loss: 0.6013, Test MSE: 0.7882, Test R2: 0.8724, Test PCC: 0.9331, Test delta R2: 0.3834, Test delta PCC: 0.6227

Processing drug_split2 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  drug_split2/test: processed 12,800/86,935 rows
  drug_split2/test: processed 25,600/86,935 rows
  drug_split2/test: processed 38,400/86,935 rows
  drug_split2/test: processed 51,200/86,935 rows
  drug_split2/test: processed 64,000/86,935 rows
  drug_split2/test: processed 76,800/86,935 rows
  drug_split2/test: processed 86,935/86,935 rows
drug_split2 / test | Test Loss: 0.6288, Test MSE: 0.8165, Test R2: 0.8675, Test PCC: 0.9305, Test delta R2: 0.3856, Test delta PCC: 0.6257

Processing drug_split3 / test


...

/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  drug_split4/test: processed 12,800/77,889 rows
  drug_split4/test: processed 25,600/77,889 rows
  drug_split4/test: processed 38,400/77,889 rows
  drug_split4/test: processed 51,200/77,889 rows
  drug_split4/test: processed 64,000/77,889 rows
  drug_split4/test: processed 76,800/77,889 rows
  drug_split4/test: processed 77,889/77,889 rows
drug_split4 / test | Test Loss: 0.6208, Test MSE: 0.8077, Test R2: 0.8701, Test PCC: 0.9318, Test delta R2: 0.3841, Test delta PCC: 0.6230

Processing drug_split5 / test


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  drug_split5/test: processed 12,800/80,260 rows
  drug_split5/test: processed 25,600/80,260 rows
  drug_split5/test: processed 38,400/80,260 rows
  drug_split5/test: processed 51,200/80,260 rows
  drug_split5/test: processed 64,000/80,260 rows
  drug_split5/test: processed 76,800/80,260 rows
  drug_split5/test: processed 80,260/80,260 rows
drug_split5 / test | Test Loss: 0.6115, Test MSE: 0.7989, Test R2: 0.8710, Test PCC: 0.9324, Test delta R2: 0.3865, Test delta PCC: 0.6247

PASS: test prediction extraction completed.
Statistics CSV:  /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_test.csv
Statistics JSON: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_test.json


,split_type,subset,n_rows,checkpoint_epoch,checkpoint_best_val_mse,output_status,h5,loss,mse,r2,pcc,delta_r2,delta_pcc
0,drug_split1,test,85539,182,0.795930,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.601333,0.788150,0.872377,0.933103,0.383421,0.622723
1,drug_split2,test,86935,192,0.789793,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.628782,0.816489,0.867488,0.930474,0.385639,0.625689
2,drug_split3,test,78097,71,0.836313,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.607424,0.793607,0.871799,0.932791,0.381960,0.620608
3,drug_split4,test,77889,192,0.804894,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.620839,0.807731,0.870083,0.931758,0.384115,0.622971
4,drug_split5,test,80260,192,0.814698,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.611527,0.798929,0.870989,0.932355,0.386459,0.624673


## Validation random-split predictions

Runs only the five `random-split_split1`–`random-split_split5` folds for the `valid` subset, writes their predicted DGE files, and prints the `prediction.py`-style performance statistics.

In [7]:
# VALIDATION RANDOM-SPLIT PREDICTIONS
valid_random_config = {
    **RANDOM_SPLIT_CONFIG,
    "subset": "valid",
}

valid_random_statistics = run_one_subset_prediction_extraction(valid_random_config)
valid_random_statistics

Using device: cuda
Loading source inputs and applying training-time total normalization...

Processing random_split1 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  random_split1/valid: processed 12,800/83,665 rows
  random_split1/valid: processed 25,600/83,665 rows
  random_split1/valid: processed 38,400/83,665 rows
  random_split1/valid: processed 51,200/83,665 rows
  random_split1/valid: processed 64,000/83,665 rows
  random_split1/valid: processed 76,800/83,665 rows
  random_split1/valid: processed 83,665/83,665 rows
random_split1 / valid | Test Loss: 0.5828, Test MSE: 0.7752, Test R2: 0.8741, Test PCC: 0.9340, Test delta R2: 0.4087, Test delta PCC: 0.6412

Processing random_split2 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  random_split2/valid: processed 12,800/83,665 rows
  random_split2/valid: processed 25,600/83,665 rows
  random_split2/valid: processed 38,400/83,665 rows
  random_split2/valid: processed 51,200/83,665 rows
  random_split2/valid: processed 64,000/83,665 rows
  random_split2/valid: processed 76,800/83,665 rows
  random_split2/valid: processed 83,665/83,665 rows
random_split2 / valid | Test Loss: 0.5886, Test MSE: 0.7808, Test R2: 0.8736, Test PCC: 0.9337, Test delta R2: 0.4068, Test delta PCC: 0.6406

Processing random_split3 / valid


...

/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  random_split4/valid: processed 12,800/83,665 rows
  random_split4/valid: processed 25,600/83,665 rows
  random_split4/valid: processed 38,400/83,665 rows
  random_split4/valid: processed 51,200/83,665 rows
  random_split4/valid: processed 64,000/83,665 rows
  random_split4/valid: processed 76,800/83,665 rows
  random_split4/valid: processed 83,665/83,665 rows
random_split4 / valid | Test Loss: 0.5839, Test MSE: 0.7756, Test R2: 0.8736, Test PCC: 0.9338, Test delta R2: 0.4047, Test delta PCC: 0.6390

Processing random_split5 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  random_split5/valid: processed 12,800/83,665 rows
  random_split5/valid: processed 25,600/83,665 rows
  random_split5/valid: processed 38,400/83,665 rows
  random_split5/valid: processed 51,200/83,665 rows
  random_split5/valid: processed 64,000/83,665 rows
  random_split5/valid: processed 76,800/83,665 rows
  random_split5/valid: processed 83,665/83,665 rows
random_split5 / valid | Test Loss: 0.5920, Test MSE: 0.7838, Test R2: 0.8729, Test PCC: 0.9334, Test delta R2: 0.4055, Test delta PCC: 0.6396

PASS: valid prediction extraction completed.
Statistics CSV:  /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_valid.csv
Statistics JSON: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_valid.json


,split_type,subset,n_rows,checkpoint_epoch,checkpoint_best_val_mse,output_status,h5,loss,mse,r2,pcc,delta_r2,delta_pcc
0,random_split1,valid,83665,191,0.775185,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.582835,0.775185,0.874125,0.934041,0.408685,0.641169
1,random_split2,valid,83665,192,0.780796,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.588614,0.780796,0.873622,0.933696,0.406827,0.640609
2,random_split3,valid,83665,191,0.773551,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.581060,0.773551,0.874334,0.934182,0.409031,0.641639
3,random_split4,valid,83665,200,0.775581,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.583883,0.775581,0.873640,0.933843,0.404661,0.638995
4,random_split5,valid,83665,192,0.783841,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.591955,0.783841,0.872944,0.933364,0.405546,0.639621


## Validation cell-split predictions

Runs only the five `cell-split_split1`–`cell-split_split5` folds for the `valid` subset, writes their predicted DGE files, and prints the `prediction.py`-style performance statistics.

In [8]:
# VALIDATION CELL-SPLIT PREDICTIONS
valid_cell_config = {
    **CELL_SPLIT_CONFIG,
    "subset": "valid",
}

valid_cell_statistics = run_one_subset_prediction_extraction(valid_cell_config)
valid_cell_statistics

Using device: cuda
Loading source inputs and applying training-time total normalization...

Processing cell_split1 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  cell_split1/valid: processed 12,800/191,127 rows
  cell_split1/valid: processed 25,600/191,127 rows
  cell_split1/valid: processed 38,400/191,127 rows
  cell_split1/valid: processed 51,200/191,127 rows
  cell_split1/valid: processed 64,000/191,127 rows
  cell_split1/valid: processed 76,800/191,127 rows
  cell_split1/valid: processed 89,600/191,127 rows
  cell_split1/valid: processed 102,400/191,127 rows
  cell_split1/valid: processed 115,200/191,127 rows
  cell_split1/valid: processed 128,000/191,127 rows
  cell_split1/valid: processed 140,800/191,127 rows
  cell_split1/valid: processed 153,600/191,127 rows
  cell_split1/valid: processed 166,400/191,127 rows
  cell_split1/valid: processed 179,200/191,127 rows
  cell_split1/valid: processed 191,127/191,127 rows
cell_split1 / valid | Test Loss: 0.7509, Test MSE: 0.9200, Test R2: 0.8552, Test PCC: 0.9236, Test delta R2: 0.2824, Test delta PCC: 0.5638

Processing cell_split2 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  cell_split2/valid: processed 12,800/80,038 rows
  cell_split2/valid: processed 25,600/80,038 rows
  cell_split2/valid: processed 38,400/80,038 rows
  cell_split2/valid: processed 51,200/80,038 rows
  cell_split2/valid: processed 64,000/80,038 rows
  cell_split2/valid: processed 76,800/80,038 rows
  cell_split2/valid: processed 80,038/80,038 rows
cell_split2 / valid | Test Loss: 0.7910, Test MSE: 0.9667, Test R2: 0.8521, Test PCC: 0.9225, Test delta R2: 0.3232, Test delta PCC: 0.5857

Processing cell_split3 / valid


...

/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  cell_split4/valid: processed 12,800/27,567 rows
  cell_split4/valid: processed 25,600/27,567 rows
  cell_split4/valid: processed 27,567/27,567 rows
cell_split4 / valid | Test Loss: 0.8302, Test MSE: 0.9991, Test R2: 0.8336, Test PCC: 0.9122, Test delta R2: 0.2875, Test delta PCC: 0.5629

Processing cell_split5 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  cell_split5/valid: processed 12,800/159,648 rows
  cell_split5/valid: processed 25,600/159,648 rows
  cell_split5/valid: processed 38,400/159,648 rows
  cell_split5/valid: processed 51,200/159,648 rows
  cell_split5/valid: processed 64,000/159,648 rows
  cell_split5/valid: processed 76,800/159,648 rows
  cell_split5/valid: processed 89,600/159,648 rows
  cell_split5/valid: processed 102,400/159,648 rows
  cell_split5/valid: processed 115,200/159,648 rows
  cell_split5/valid: processed 128,000/159,648 rows
  cell_split5/valid: processed 140,800/159,648 rows
  cell_split5/valid: processed 153,600/159,648 rows
  cell_split5/valid: processed 159,648/159,648 rows
cell_split5 / valid | Test Loss: 0.7234, Test MSE: 0.8935, Test R2: 0.8515, Test PCC: 0.9222, Test delta R2: 0.3050, Test delta PCC: 0.5671

PASS: valid prediction extraction completed.
Statistics CSV:  /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/pred

,split_type,subset,n_rows,checkpoint_epoch,checkpoint_best_val_mse,output_status,h5,loss,mse,r2,pcc,delta_r2,delta_pcc
0,cell_split1,valid,191127,96,0.920025,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.750883,0.920025,0.855199,0.923639,0.282386,0.563807
1,cell_split2,valid,80038,88,0.966656,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.790956,0.966656,0.852142,0.922545,0.323234,0.585665
2,cell_split3,valid,61134,45,0.914361,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.747971,0.914361,0.850633,0.921809,0.271467,0.554634
3,cell_split4,valid,27567,36,0.999096,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.830238,0.999096,0.833643,0.912204,0.287537,0.562859
4,cell_split5,valid,159648,85,0.893537,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.723411,0.893537,0.851477,0.922163,0.304995,0.567086


## Validation drug-split predictions

Runs only the five `drug-split_split1`–`drug-split_split5` folds for the `valid` subset, writes their predicted DGE files, and prints the `prediction.py`-style performance statistics.

In [9]:
# VALIDATION DRUG-SPLIT PREDICTIONS
valid_drug_config = {
    **DRUG_SPLIT_CONFIG,
    "subset": "valid",
}

valid_drug_statistics = run_one_subset_prediction_extraction(valid_drug_config)
valid_drug_statistics

Using device: cuda
Loading source inputs and applying training-time total normalization...

Processing drug_split1 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  drug_split1/valid: processed 12,800/90,277 rows
  drug_split1/valid: processed 25,600/90,277 rows
  drug_split1/valid: processed 38,400/90,277 rows
  drug_split1/valid: processed 51,200/90,277 rows
  drug_split1/valid: processed 64,000/90,277 rows
  drug_split1/valid: processed 76,800/90,277 rows
  drug_split1/valid: processed 89,600/90,277 rows
  drug_split1/valid: processed 90,277/90,277 rows
drug_split1 / valid | Test Loss: 0.6092, Test MSE: 0.7959, Test R2: 0.8695, Test PCC: 0.9315, Test delta R2: 0.3841, Test delta PCC: 0.6225

Processing drug_split2 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  drug_split2/valid: processed 12,800/82,583 rows
  drug_split2/valid: processed 25,600/82,583 rows
  drug_split2/valid: processed 38,400/82,583 rows
  drug_split2/valid: processed 51,200/82,583 rows
  drug_split2/valid: processed 64,000/82,583 rows
  drug_split2/valid: processed 76,800/82,583 rows
  drug_split2/valid: processed 82,583/82,583 rows
drug_split2 / valid | Test Loss: 0.6020, Test MSE: 0.7898, Test R2: 0.8718, Test PCC: 0.9327, Test delta R2: 0.3877, Test delta PCC: 0.6261

Processing drug_split3 / valid


...

/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  drug_split4/valid: processed 12,800/83,510 rows
  drug_split4/valid: processed 25,600/83,510 rows
  drug_split4/valid: processed 38,400/83,510 rows
  drug_split4/valid: processed 51,200/83,510 rows
  drug_split4/valid: processed 64,000/83,510 rows
  drug_split4/valid: processed 76,800/83,510 rows
  drug_split4/valid: processed 83,510/83,510 rows
drug_split4 / valid | Test Loss: 0.6178, Test MSE: 0.8049, Test R2: 0.8691, Test PCC: 0.9313, Test delta R2: 0.3860, Test delta PCC: 0.6237

Processing drug_split5 / valid


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


  drug_split5/valid: processed 12,800/94,518 rows
  drug_split5/valid: processed 25,600/94,518 rows
  drug_split5/valid: processed 38,400/94,518 rows
  drug_split5/valid: processed 51,200/94,518 rows
  drug_split5/valid: processed 64,000/94,518 rows
  drug_split5/valid: processed 76,800/94,518 rows
  drug_split5/valid: processed 89,600/94,518 rows
  drug_split5/valid: processed 94,518/94,518 rows
drug_split5 / valid | Test Loss: 0.6269, Test MSE: 0.8147, Test R2: 0.8629, Test PCC: 0.9281, Test delta R2: 0.3871, Test delta PCC: 0.6258

PASS: valid prediction extraction completed.
Statistics CSV:  /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_valid.csv
Statistics JSON: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge/statistics/prediction_py_style_statistics_valid.json

,split_type,subset,n_rows,checkpoint_epoch,checkpoint_best_val_mse,output_status,h5,loss,mse,r2,pcc,delta_r2,delta_pcc
0,drug_split1,valid,90277,182,0.795930,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.609175,0.795930,0.869463,0.931532,0.384129,0.622515
1,drug_split2,valid,82583,192,0.789793,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.601976,0.789793,0.871781,0.932737,0.387737,0.626060
2,drug_split3,valid,88185,71,0.836313,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.648318,0.836313,0.862124,0.927631,0.387636,0.626649
3,drug_split4,valid,83510,192,0.804894,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.617791,0.804894,0.869078,0.931276,0.386034,0.623678
4,drug_split5,valid,94518,192,0.814698,WRITTEN,/work/users/m/e/meisheng/Dissertation/Experime...,0.626945,0.814698,0.862905,0.928080,0.387118,0.625842
